# آموزش TTS فارسی روی Google Colab — تست دود مسیر کامل (TTV-v1)

> ⚠️ **این نوت‌بوک برای شاخهٔ `codex` (espeak/۱۷۸) است و با پروژهٔ واقعی شما `perfect_project` (کاراکتری/۲۴۶) سازگار نیست.**
> از نوت‌بوک **`FA_TTV_Colab_perfect_project.ipynb`** استفاده کنید.

این نوت‌بوک کل مسیر را روی **بخشی از داده‌ها** با GPU Colab اجرا می‌کند تا قبل از اجارهٔ GPU گران، مطمئن شویم همه‌چیز کار می‌کند. خروجی هر مرحله روی **Google Drive** ذخیره می‌شود.

سلول‌ها را به‌ترتیب اجرا کنید.

---
### نکتهٔ تعیین‌کننده دربارهٔ چک‌پوینت (حتماً بخوانید)
معماری مدل به اندازهٔ واژگان (`n_vocab`) وابسته است (لایهٔ `enc_p.emb` و سر `phoneme_classifier`).

- اگر چک‌پوینت `G_3135000` با مدل **۱۷۸تایی استاندارد** آموزش دیده باشد → با کد همین نوت‌بوک **سازگار است و می‌توان ادامه داد**.
- اگر با جدول نماد **۲۴۶تایی** سفارشی (شاخهٔ fix شما) آموزش دیده باشد → با این کد **لود نمی‌شود**؛ یا باید کد ۲۴۶تایی خودتان را استفاده کنید یا از صفر شروع کنید.

**سلول «بازرسی چک‌پوینت» (شمارهٔ ۸) این موضوع را با شواهد مشخص می‌کند. حتماً قبل از تصمیمِ ادامه/از‌صفر آن را اجرا کنید.**

> اگر دسترسی لینک‌های Drive روی «هرکسی با لینک» نباشد، `gdown` خطا می‌دهد. در آن صورت یا اشتراک را فعال کنید، یا فایل‌ها را در `MyDrive` خودتان بگذارید و از مسیر mount‌شده کپی کنید.

In [ ]:
# [1] بررسی GPU (مطمئن شوید Runtime > Change runtime type > GPU فعال است)
!nvidia-smi

In [ ]:
# [2] اتصال Google Drive و تعیین پوشهٔ خروجی (همهٔ نتایج اینجا ذخیره می‌شود)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = "/content/drive/MyDrive/zero_tts_run"   # در صورت نیاز تغییر دهید
os.makedirs(DRIVE_OUT, exist_ok=True)
print("outputs ->", DRIVE_OUT)

In [ ]:
# [3] دریافت کد + وصله‌های لازم برای اجرای بی‌خطا روی Colab
%cd /content
!rm -rf hier
!git clone https://github.com/rt4439582-afk/HierSpeechpp.git hier
%cd /content/hier
# شاخه‌ای که persian_cleaners و اسکریپت‌های آماده‌سازی فارسی را دارد:
!git checkout codex/provide-fine-tuning-steps-for-persian-data

import pathlib
# وصله ۱: افزودن import گمشدهٔ monotonic_align (وگرنه در forward آموزش NameError می‌دهد)
tf = pathlib.Path("ttv_v1/t2w2v_transformer.py"); s = tf.read_text()
if "import monotonic_align" not in s:
    s = s.replace("import attentions", "import attentions\nimport monotonic_align", 1)
    tf.write_text(s); print("patched: added 'import monotonic_align'")
# وصله ۲: کاهش worker‌ها (کد ۳۲ را هاردکد کرده؛ برای Colab زیاد است)
tr = pathlib.Path("train_ttv_v1.py"); t = tr.read_text()
if "num_workers=32" in t:
    tr.write_text(t.replace("num_workers=32", "num_workers=2")); print("patched: num_workers=2")

In [ ]:
# [4] نصب وابستگی‌ها (از torch پیش‌نصب Colab استفاده می‌کنیم؛ فقط بقیه را نصب می‌کنیم)
!apt-get -qq update && apt-get -qq install -y espeak-ng
!pip -q install gdown "phonemizer==3.2.1" Unidecode einops amfm_decompy librosa
# تست سریع G2P فارسی espeak:
!espeak-ng -v fa "سلام دنیا" --ipa || echo "espeak-ng fa not OK"

In [ ]:
# [5] build کردن monotonic_align (نسخهٔ .so داخل مخزن برای پایتون قدیمی است) + تنظیم PYTHONPATH
%cd /content/hier/ttv_v1/monotonic_align
!python setup.py build_ext --inplace
%cd /content/hier
import os
# تا زیرفرایند `python train_ttv_v1.py` بتواند `import monotonic_align` کند:
os.environ["PYTHONPATH"] = "/content/hier/ttv_v1:" + os.environ.get("PYTHONPATH", "")
print("PYTHONPATH =", os.environ["PYTHONPATH"])

In [ ]:
# [6] دانلود چک‌پوینت‌ها و داده‌ها از Google Drive
import os
os.makedirs("/content/ckpts", exist_ok=True)
os.makedirs("/content/data_zips", exist_ok=True)

CKPTS = {
    "G_3135000.zip": "1mUDS-Zh8m8P_p6REeRUDjkn7bMDSjkPM",   # چک‌پوینت فارسی قبلی (zip)
    "G_0.pth":       "187G1lPWLa__wSeGuLrNP1IlcOXkWholc",   # چک‌پوینت پایه
}
DATA_IDS = [
    "1zA_CO5_yao5f9Gg2OwLxwdTFxHxM2YuP",
    "15vnhZ2a_dnymdYYHJ7AQktb18d-yzJ0F",
    "13gVShTSC-OM1eHeuJRjCFinHC_1LIKpj",
    "1vfRUEIO_OgZG9nij-CfCRtMTK2D__qKP",
    "1uW82gIKugERwHnynBQKNK_xmp8ZU_sm5",
    "14M8BD52Nw22imoKG_9v2atLevJun5DAv",
    "1BQT5j3Zb8tQT140aVRzzgnYmdq0UGI8L",
]

for name, fid in CKPTS.items():
    !gdown "https://drive.google.com/uc?id={fid}" -O /content/ckpts/{name}

%cd /content/data_zips
for fid in DATA_IDS:
    !gdown "https://drive.google.com/uc?id={fid}"   # با نام اصلی فایل ذخیره می‌شود
%cd /content/hier
!ls -lh /content/ckpts /content/data_zips

In [ ]:
# [7] تشخیص نوع فایل‌ها و باز کردن zipها + دیدن ساختار واقعی داده
import glob
print("== نوع فایل‌ها ==")
for f in sorted(glob.glob("/content/data_zips/*")) + sorted(glob.glob("/content/ckpts/*")):
    !file "{f}"

!mkdir -p /content/ckpts_x /content/data
for f in glob.glob("/content/ckpts/*.zip"):
    !unzip -o -q "{f}" -d /content/ckpts_x
for f in glob.glob("/content/data_zips/*.zip"):
    !unzip -o -q "{f}" -d /content/data

print("\n== چک‌پوینت‌های استخراج‌شده ==")
!find /content/ckpts_x -maxdepth 3 -name "*.pth"
print("\n== ساختار داده (۸۰ خط اول) ==")
!find /content/data -maxdepth 4 | head -80

In [ ]:
# [8] بازرسی چک‌پوینت — مهم‌ترین سلول تصمیم‌گیری
# اندازهٔ واژگانِ واقعیِ چک‌پوینت را نشان می‌دهد تا بدانید با این کد سازگار است یا نه.
import torch, glob
cands = sorted(glob.glob("/content/ckpts_x/**/G_*.pth", recursive=True)) + sorted(glob.glob("/content/ckpts/*.pth"))
print("چک‌پوینت‌های یافت‌شده:", cands)
assert cands, "هیچ G_*.pth پیدا نشد — سلول‌های ۶ و ۷ را بررسی کنید."

for ck in cands:
    d = torch.load(ck, map_location="cpu")
    sd = d["model"] if isinstance(d, dict) and "model" in d else d
    emb = next((tuple(v.shape) for k, v in sd.items() if k.endswith("enc_p.emb.weight")), None)
    cls = next((tuple(v.shape) for k, v in sd.items() if "phoneme_classifier" in k), None)
    it  = d.get("iteration") if isinstance(d, dict) else None
    print(f"\n{ck}\n  enc_p.emb.weight = {emb}   phoneme_classifier = {cls}   iteration = {it}")
    if emb:
        print("  => n_vocab این چک‌پوینت:", emb[0],
              "| سازگار با کد فعلی (۱۷۸)؟", "بله ✅" if emb[0] == 178 else "خیر ❌ (کد ۲۴۶تایی لازم است یا از صفر)")

### تنظیم مسیر داده‌ها

از خروجی سلول ۷ ساختار واقعی دیتای خود را ببینید و در سلول بعدی مسیرها را اصلاح کنید:

- اگر داده‌ها **خام** هستند (`*.wav` ۱۶kHz + `*.txt` هم‌نام) → سلول‌های ۱۰ تا ۱۳ را اجرا کنید (ساخت زیرمجموعه، استخراج ویژگی، filelist).
- اگر داده‌ها **از قبل پیش‌پردازش** شده‌اند (شامل `f0/ w2v/ token` به‌صورت `*.pt` و فایل‌های `train_*.txt`) → سلول‌های ۱۰ تا ۱۲ را رد کنید و در سلول ۱۳ فقط `--output_dir` را به مسیر filelist موجود اشاره دهید یا مستقیم در سلول ۱۴ مسیر filelist را تنظیم کنید.

> توکنایزر را روی `persian_cleaners` نگه می‌داریم. اگر چک‌پوینت شما (سلول ۸) `n_vocab=178` بود، این مسیر برای **ادامهٔ آموزش** درست است.

In [ ]:
# [10] ساخت زیرمجموعهٔ کوچک برای تست دود (فقط اگر داده خام wav+txt است)
# مسیرها را مطابق خروجی سلول ۷ اصلاح کنید:
RAW_WAV = "/content/data/wav16k"   # <-- اصلاح کنید
RAW_TXT = "/content/data/text"     # <-- اصلاح کنید
NUM_SAMPLES = 200

!python scripts/make_smoke_subset.py \
    --src_wav_dir "{RAW_WAV}" --src_text_dir "{RAW_TXT}" \
    --dst_root /content/smoke --num_samples {NUM_SAMPLES}
!echo "---"; find /content/smoke -maxdepth 2 | head

In [ ]:
# [11] استخراج ویژگی‌ها (w2v با GPU، F0 با YAAPT، token با persian_cleaners)
!python ttv_v1/preprocessing/extract_w2v_generic.py \
    --input_wav_dir /content/smoke/wav16k --output_w2v_dir /content/smoke/w2v --device cuda
!python ttv_v1/preprocessing/extract_f0_generic.py \
    --input_wav_dir /content/smoke/wav16k --output_f0_dir /content/smoke/f0
!python ttv_v1/preprocessing/extract_token_generic.py \
    --input_text_dir /content/smoke/text --output_token_dir /content/smoke/token --cleaner persian_cleaners

In [ ]:
# [12] ساخت filelist + اعتبارسنجی دیتاست
!python ttv_v1/preprocessing/prepare_filelist_generic.py \
    --wav_dir /content/smoke/wav16k --f0_dir /content/smoke/f0 \
    --token_dir /content/smoke/token --w2v_dir /content/smoke/w2v \
    --output_dir filelists/fa_smoke
!python scripts/validate_fa_dataset.py \
    --wav_dir /content/smoke/wav16k --text_dir /content/smoke/text \
    --report_csv /content/smoke/validate_report.csv \
    --summary_json /content/smoke/validate_summary.json || true
print("--- نمونهٔ train_wav.txt ---")
!head -n 3 filelists/fa_smoke/train_wav.txt
!wc -l filelists/fa_smoke/train_*.txt

In [ ]:
# [13] ساخت config کوچک برای تست دود (بر پایهٔ config_fa_template، با persian_cleaners)
import json, pathlib
cfg = json.load(open("ttv_v1/config_fa_template.json"))
cfg["train"].update({"epochs": 3, "batch_size": 2, "num_workers": 2,
                     "log_interval": 1, "eval_interval": 10, "save_interval": 10,
                     "fp16_run": False})
cfg["data"].update({"train_filelist_path": "filelists/fa_smoke/train_wav.txt",
                    "test_filelist_path":  "filelists/fa_smoke/train_wav.txt",
                    "text_cleaners": ["persian_cleaners"]})
cfg["data"].setdefault("train_data_ratio", 1.0)
pathlib.Path("ttv_v1/config_fa_smoke.json").write_text(json.dumps(cfg, ensure_ascii=False, indent=2))
print("نوشته شد: ttv_v1/config_fa_smoke.json")
print(json.dumps(cfg["train"], ensure_ascii=False, indent=2))

In [ ]:
# [14] (اختیاری) برای «ادامه از چک‌پوینت قبلی»: کپی چک‌پوینت سازگار در پوشهٔ لاگ
# فقط اگر سلول ۸ نشان داد n_vocab = 178 (سازگار) است این سلول را اجرا کنید.
# برای «شروع از صفر»، این سلول را اجرا نکنید.
import glob, shutil, os
os.makedirs("logs/ttv_fa_smoke", exist_ok=True)
cands = sorted(glob.glob("/content/ckpts_x/**/G_*.pth", recursive=True)) + sorted(glob.glob("/content/ckpts/*.pth"))
if cands:
    src = cands[0]
    dst = os.path.join("logs/ttv_fa_smoke", os.path.basename(src))
    shutil.copy(src, dst)
    print("ادامه از:", dst)
else:
    print("چک‌پوینتی کپی نشد — از صفر آموزش می‌بیند.")

In [ ]:
# [15] اجرای چند گام آموزش (تست دود)
import os
os.environ["PYTHONPATH"] = "/content/hier/ttv_v1:" + os.environ.get("PYTHONPATH", "")
!CUDA_VISIBLE_DEVICES=0 python train_ttv_v1.py -c ttv_v1/config_fa_smoke.json -m ttv_fa_smoke
# معیار موفقیت: بدون خطا اجرا شود، loss متناهی باشد (NaN/Inf نه)، و فایل logs/ttv_fa_smoke/G_*.pth ذخیره شود.

In [ ]:
# [16] ذخیرهٔ خروجی‌ها روی Google Drive
!mkdir -p "{DRIVE_OUT}/logs" "{DRIVE_OUT}/filelists"
!rsync -a logs/ttv_fa_smoke "{DRIVE_OUT}/logs/"
!cp -r filelists/fa_smoke "{DRIVE_OUT}/filelists/"
!cp ttv_v1/config_fa_smoke.json "{DRIVE_OUT}/"
print("ذخیره شد در:", DRIVE_OUT)
!ls -lh "{DRIVE_OUT}/logs/ttv_fa_smoke"

## بعد از تست دود موفق

اگر سلول ۱۵ بدون خطا اجرا شد، `loss` متناهی بود و چک‌پوینت ذخیره شد → مسیر سالم است.

### آموزش کامل (روی GPU اجاره‌ای ~۸ روز)
۱. **کل** داده‌ها را پیش‌پردازش کنید (سلول‌های ۱۰–۱۲ روی کل دیتاست، نه فقط ۲۰۰ نمونه).
۲. configِ بزرگ بسازید: `batch_size` ۸–۱۶، `save_interval`/`eval_interval` ۱۰۰۰۰، `epochs` زیاد.
۳. برای ادامه از مدل قبلی، چک‌پوینت `G_3135000` را در `logs/<exp>/` بگذارید (سلول ۱۴).
۴. آموزش را داخل `tmux`/`nohup` اجرا کنید و **هر چند ساعت** `logs/` را با `rsync` روی Drive بک‌آپ بگیرید (ماشین اجاره‌ای ممکن است پاک شود).
۵. پایش با TensorBoard: `loss/g/ctc` باید پایین بیاید (الایمنت)، `loss/g/w2v` کاهش یابد.

### inference نهایی
TTV خروجی w2v می‌دهد؛ برای صدا، آن را به `hierspeechpp_speechsynthesizer` و سپس `SpeechSR` بدهید (به `inference.py` نگاه کنید).

> راهنمای کامل متنی: `docs/FA_TTV_RUNBOOK.md`